# Комплексная обработка датафреймов
В рабочей папке лежит файл itresume-coderun.csv. В нем содержится кусочек информации о том, как пользователи платформы IT Resume отправляют код на выполнение (нажимают на кнопку Проверить):


- идентификатор записи

- дата и время отправки кода на выполнение

- идентификатор задачи

- идентификатор пользователя

- идентификатор языка

Также в вашей рабочей папке лежит файл itresume-users-pandas.csv. В нем содержится зашифрованная информация о некоторых пользователях платформы IT Resume:


- идентификатор пользователя в базе данных

- зашифрованный логин пользователя

- дата регистрации на платформе


## Задание
Загрузите файлы в датафреймы _Pandas_ и назовите их *coderun* и *users*. Далее создайте новый датафрейм _df_, который получается в результате:

- Объединения двух датафреймов по полям с пользователем
- Если для каких-то кодранов не найдена пара в таблице _users_, такие записи нужно все равно оставить в результирующей таблице
- Если есть совпадающие названия столбцов, то у них в результате должны быть суффиксы *_run* и *_user*

Далее выполните действия:

- Создайте новый датафрейм _df2_, который будет хранить информацию о том, сколько дней прошло между последним выполнением кода конкретного юзера и днем его регистрации. Столбец с разницей должен называться _diff_.
- Округлите diff до ближайшего меньшего числа, кратного 50. Результат должен остаться целым числом.
- Создайте датафрейм _df3_, в который запишите количество юзеров, сгруппированных по разнице дней.
- Результат отсортируйте по убыванию количества *user_id*.

### Не забудьте, что пропуски тоже нужно учитывать при расчетах!

In [1]:
import pandas as pd

In [3]:
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
coderun = pd.read_csv('D:/GitHub_projects/Simulative_course/data/itresume-coderun.csv', encoding='1251')
users = pd.read_csv('D:/GitHub_projects/Simulative_course/data/itresume-users-pandas.csv', encoding='1251', sep=';', skiprows=1)


In [4]:
coderun.head()

,id,created_at,problem_id,user_id,language_id
0,1,2021-04-07 06:06:20.000,13,10,3
1,2,2021-03-31 07:10:06.000,15,13,3
2,3,2021-04-04 14:55:26.000,1,6,3
3,4,2021-03-29 21:24:51.000,26,4,3
4,5,2021-03-30 11:29:12.000,21,18,3


In [5]:
users.head()

,id,username,date_joined
0,416,nsosabisfsobm,2022-01-02 12:22:43
1,43,NbabnnbTona,2021-07-01 09:11:42
2,50,jbdn,2021-08-04 09:01:56
3,249,Kbaoxsnb,2021-11-27 07:03:35
4,77,bxakbasmo$$$,2021-11-09 14:31:46


In [94]:
df = coderun.merge(users, left_on='user_id', right_on='id', how='left', suffixes=('_run', '_user'))
df

,id_run,created_at,problem_id,user_id,language_id,id_user,username,date_joined
0,1,2021-04-07 06:06:20.000,13,10,3,NaN,NaN,NaN
1,2,2021-03-31 07:10:06.000,15,13,3,NaN,NaN,NaN
2,3,2021-04-04 14:55:26.000,1,6,3,NaN,NaN,NaN
3,4,2021-03-29 21:24:51.000,26,4,3,NaN,NaN,NaN
4,5,2021-03-30 11:29:12.000,21,18,3,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
55717,55718,2022-05-17 09:00:51.743,101,171,2,NaN,NaN,NaN
55718,55719,2022-05-17 09:01:04.210,101,171,2,NaN,NaN,NaN
55719,55720,2022-05-17 09:02:06.203,101,171,2,NaN,NaN,NaN
55720,55721,2022-05-17 09:03:45.265,102,171,2,NaN,NaN,NaN


In [106]:
df_diff = df[['user_id', 'created_at', 'date_joined']].copy()
df_diff['created_at'] = pd.to_datetime(df_diff['created_at'])
df_diff['date_joined'] = pd.to_datetime(df_diff['date_joined'])
df2 = df_diff.groupby('user_id')[['created_at','date_joined']].agg({'created_at': 'max', 'date_joined' : 'min'}).reset_index()
df2['diff']  = (df2['created_at'] - df2['date_joined']).dt.days
df2['diff'] = ((df2['diff']//50)*50).astype('Int64')
df2 = df2[['user_id', 'diff']]
df2

,user_id,diff
0,1,<NA>
1,2,<NA>
2,3,<NA>
3,4,<NA>
4,5,<NA>
...,...,...
863,2804,<NA>
864,2812,<NA>
865,2813,<NA>
866,2814,<NA>


In [107]:
df2

,user_id,diff
0,1,<NA>
1,2,<NA>
2,3,<NA>
3,4,<NA>
4,5,<NA>
...,...,...
863,2804,<NA>
864,2812,<NA>
865,2813,<NA>
866,2814,<NA>


In [108]:
df3 = df2.groupby('diff', dropna=False)['user_id'].agg('nunique').sort_values(ascending=False).reset_index()
df3

,diff,user_id
0,<NA>,795
1,0,41
2,100,16
3,50,11
4,150,3
5,250,2
